# 🥉 **Ingesta de Datos - Capa Bronze**
*Datos crudos tal como vienen de la fuente, sin transformaciones.*

In [0]:
from databricks.connect import DatabricksSession
from pyspark.sql import DataFrame
from pyspark.sql.utils import AnalysisException
import logging

# --- Configuración de logging -------------------------------------------------
logging.basicConfig(level=logging.INFO, format = '%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# --- Constantes --------------------------------------------------------------- 
BASE_PATH = '/Workspace/Users/raul.perez.costero@gmail.com/.bundle/Data Engineer/dev/files/databriks-repositorio/notebooks'

CATALOG = 'workspace'

SCHEMA = 'retail_db'

SOURCES = {
    "orders":   "olist_orders_dataset.csv",
    "items":    "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
} 

# --- Helpers -------------------------------------------------------------------
def read_csv(spark: DatabricksSession, path: str) -> DataFrame:
    '''Lee un CSV con cabecera y lo devuelve como un DataFrame'''
    return (
        spark.read
        .format('csv')
        .options(header = True, inferSchema=True)
        .load(path)
        )

def save_as_bronze(df: DataFrame, table: str) -> None:
    '''Escribe un DataFrame como un Delta Lake'''
    full_table = f'{CATALOG}.{SCHEMA}.{table}'
    (
        df.write
        .format('delta')
        .mode('overwrite')
        .options(overwriteSchema = True)
        .saveAsTable(full_table)
     )
    logging.info(f'\33[35mTabla {full_table} guardada correctamente.\33[0m')


# --- Pipeline principal ---------------------------------------------------------
def main() -> None:
    spark = DatabricksSession.builder.getOrCreate()

    # 1. Crear schema si no existe
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    logger.info(f"\33[36mSchema '{CATALOG}.{SCHEMA}' verificado.\33[0m")

    # 2. Leer → escribir cada fuente
    for table_name, filename in SOURCES.items():
        path = f"{BASE_PATH}/{filename}"
        try:
            logger.info(f"\33[33mLeyendo {filename}...\33[0m")
            df = read_csv(spark, path)
            save_as_bronze(df, table_name)
        except AnalysisException as e:
            logger.error(f"\33[31mError procesando '{filename}': {e}\33[0m")
            raise

    logger.info(f"\33[32m✅ Tablas Bronze creadas correctamente en {CATALOG}.{SCHEMA}✅\33[0m")

if __name__ == "__main__":
    main()


In [0]:
%sql
SELECT * FROM workspace.retail_db.items LIMIT 3

# 🥈 **Transformación de Datos - Capa Silver**
*Datos limpios, validados y estructurados listos para analizar.*

In [0]:
from databricks.connect import DatabricksSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, to_timestamp, datediff, when, round as spark_round,
    trim, upper, regexp_replace, current_timestamp
)
from pyspark.sql.utils import AnalysisException
import logging

# --- Configuración de logging -------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# --- Constantes ---------------------------------------------------------------
CATALOG = 'workspace'
BRONZE_SCHEMA = 'retail_db'
SILVER_SCHEMA = 'retail_db_silver'

TIMESTAMP_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

# --- Helpers ------------------------------------------------------------------
def read_bronze(spark: DatabricksSession, table: str) -> DataFrame:
    """Lee una tabla Delta de la capa Bronze."""
    full_table = f'{CATALOG}.{BRONZE_SCHEMA}.{table}'
    logger.info(f"\33[33mLeyendo tabla bronze: {full_table}\33[0m")
    return spark.read.format('delta').table(full_table)


def save_as_silver(df: DataFrame, table: str) -> None:
    """Escribe un DataFrame como tabla Delta en la capa Silver."""
    full_table = f'{CATALOG}.{SILVER_SCHEMA}.{table}'
    (
        df.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', True)
        .saveAsTable(full_table)
    )
    logger.info(f"\33[35mTabla {full_table} guardada correctamente.\33[0m")


# --- Transformaciones por entidad ---------------------------------------------
def clean_orders(df: DataFrame) -> DataFrame:
    """
    Limpieza y enriquecimiento de orders:
    - Castea timestamps
    - Elimina filas sin order_id o customer_id
    - Filtra estados válidos
    - Calcula días reales de entrega y desviación vs estimación
    """
    valid_statuses = ['delivered', 'shipped', 'invoiced', 'processing',
                      'approved', 'unavailable', 'canceled', 'created']

    # Castear timestamps
    df_typed = df
    for ts_col in TIMESTAMP_COLS:
        if ts_col in df.columns:
            df_typed = df_typed.withColumn(ts_col, to_timestamp(col(ts_col)))

    return (
        df_typed
        # Eliminar nulos críticos
        .dropna(subset=['order_id', 'customer_id'])
        # Eliminar duplicados
        .dropDuplicates(['order_id'])
        # Filtrar estados conocidos
        .filter(col('order_status').isin(valid_statuses))
        # Días reales de entrega (purchase → delivered)
        .withColumn(
            'delivery_days',
            when(
                col('order_delivered_customer_date').isNotNull(),
                datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp'))
            )
        )
        # Desviación respecto a la estimación (negativo = antes, positivo = tarde)
        .withColumn(
            'delivery_delay_days',
            when(
                col('order_delivered_customer_date').isNotNull() &
                col('order_estimated_delivery_date').isNotNull(),
                datediff(col('order_delivered_customer_date'), col('order_estimated_delivery_date'))
            )
        )
        .withColumn('silver_loaded_at', current_timestamp())
    )


def clean_items(df: DataFrame) -> DataFrame:
    """
    Limpieza de order_items:
    - Elimina nulos críticos
    - Castea tipos numéricos
    - Calcula precio total por línea (price + freight)
    """
    return (
        df
        .dropna(subset=['order_id', 'product_id'])
        .dropDuplicates(['order_id', 'order_item_id'])
        .withColumn('price',          col('price').cast('double'))
        .withColumn('freight_value',  col('freight_value').cast('double'))
        .withColumn('shipping_limit_date', to_timestamp(col('shipping_limit_date')))
        # Precio total de la línea
        .withColumn('line_total', spark_round(col('price') + col('freight_value'), 2))
        .withColumn('silver_loaded_at', current_timestamp())
    )


def clean_products(df: DataFrame) -> DataFrame:
    """
    Limpieza de products:
    - Normaliza categoría (trim, uppercase, reemplaza guiones)
    - Elimina productos sin ID
    - Castea dimensiones y peso
    """
    return (
        df
        .dropna(subset=['product_id'])
        .dropDuplicates(['product_id'])
        # Normalizar categoría
        .withColumn(
            'product_category_name',
            upper(trim(regexp_replace(col('product_category_name'), '_', ' ')))
        )
        # Castear métricas físicas
        .withColumn('product_weight_g',    col('product_weight_g').cast('double'))
        .withColumn('product_length_cm',   col('product_length_cm').cast('double'))
        .withColumn('product_height_cm',   col('product_height_cm').cast('double'))
        .withColumn('product_width_cm',    col('product_width_cm').cast('double'))
        .withColumn('silver_loaded_at', current_timestamp())
    )


# --- Pipeline principal -------------------------------------------------------
def main() -> None:
    spark = DatabricksSession.builder.getOrCreate()

    # 1. Crear schema Silver si no existe
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
    logger.info(f"\33[36mSchema '{CATALOG}.{SILVER_SCHEMA}' verificado.\33[0m")

    # 2. Mapa: tabla bronze → función de limpieza → nombre silver
    pipeline = [
        ('orders',   clean_orders,   'orders'),
        ('items',    clean_items,    'order_items'),
        ('products', clean_products, 'products'),
    ]

    # 3. Ejecutar cada etapa
    for bronze_table, transform_fn, silver_table in pipeline:
        try:
            df_bronze = read_bronze(spark, bronze_table)
            df_silver = transform_fn(df_bronze)

            # Log de registros procesados
            count = df_silver.count()
            logger.info(f"\33[33m{silver_table}: {count:,} registros limpios\33[0m")

            save_as_silver(df_silver, silver_table)

        except AnalysisException as e:
            logger.error(f"\33[31mError procesando '{bronze_table}': {e}\33[0m")
            raise

    logger.info(f"\33[32m✅ Tablas Silver creadas correctamente en {CATALOG}.{SILVER_SCHEMA} ✅\33[0m")


if __name__ == "__main__":
    main()

In [0]:
%sql
SELECT * FROM workspace.retail_db_silver.order_items LIMIT 5

# 🥇 **Modelado de Datos - Capa Gold**
*Datos agregados y optimizados para consumo de negocio.*

# 💎 **Serving Layer - Capa Platinum** 
*Métricas y KPIs finales expuestos para dashboards y reportes.*

In [0]:
## No tiene sentido hacer esta capa en un notebook, usar Power BI + Snowflake, MLflow, Databricks Model Serving, , Tableau, Looker